In [11]:
# !gcloud auth application-default login

In [12]:
import google.auth

_, project_id = google.auth.default()
print(f"Project ID: {project_id}")

Project ID: dev-mq-tech-transfer


In [ ]:
# Cell 1 — instrumentation (run once per kernel session)
from phoenix.otel import register
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

tracer_provider = register(
    project_name="gap_assessment", # set your project name here when you start the phoenix server
    endpoint="http://localhost:6006/v1/traces"
)
GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)

In [13]:
import json
import os
import re
from typing import Any, Dict, List, Tuple

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.cloud import logging as gcp_logging
from google.genai import types as genai_types


<!-- ## Chunk-Based RAG Workflow -->

In [14]:
def _proto_to_dict(obj: Any) -> Dict[str, Any]:
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj

    try:
        from google.protobuf.json_format import MessageToDict

        parsed = MessageToDict(obj._pb) if hasattr(obj, "_pb") else MessageToDict(obj)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def _extract_classification_rds(details: Dict[str, Any]) -> Any:
    if not isinstance(details, dict):
        return None

    for key in ["classification_rds", "classificationRds", "classification", "information_classification__c", "rds_classification"]:
        value = details.get(key)
        if value not in (None, ""):
            return value

    raw_json_data = details.get("jsonData") or details.get("json_data")
    if isinstance(raw_json_data, str) and raw_json_data.strip().startswith("{"):
        try:
            parsed = json.loads(raw_json_data)
            if isinstance(parsed, dict):
                for key in ["classification_rds", "classificationRds", "classification", "information_classification__c", "rds_classification"]:
                    value = parsed.get(key)
                    if value not in (None, ""):
                        return value
        except json.JSONDecodeError:
            return None

    return None


def get_document_details(project_id: str, location: str, datastore_id: str, doc_id: str, cache: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    from google.api_core.client_options import ClientOptions
    from google.cloud import discoveryengine_v1 as discoveryengine

    if not doc_id:
        return {}
    if doc_id in cache:
        return cache[doc_id]

    endpoint = "discoveryengine.googleapis.com" if location.lower() == "global" else f"{location}-discoveryengine.googleapis.com"
    client = discoveryengine.DocumentServiceClient(client_options=ClientOptions(api_endpoint=endpoint))

    name = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/branches/default_branch/documents/{doc_id}"
    )

    details: Dict[str, Any] = {}
    try:
        doc = client.get_document(name=name)
        doc_dict = _proto_to_dict(doc)

        if isinstance(doc_dict.get("structData"), dict):
            details.update(doc_dict.get("structData", {}))
        if isinstance(doc_dict.get("derivedStructData"), dict):
            details.update(doc_dict.get("derivedStructData", {}))

        uri = str(getattr(doc, "uri", "") or "").strip()
        if not uri:
            content_obj = getattr(doc, "content", None)
            uri = str(getattr(content_obj, "uri", "") or "").strip() if content_obj is not None else ""
        if uri:
            details["uri"] = uri
    except Exception:
        details = {}

    cache[doc_id] = details
    return details


def search_relevant_chunks(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    top_k: int = 8,
) -> List[Dict[str, Any]]:
    from google.api_core.client_options import ClientOptions
    from google.cloud import discoveryengine_v1 as discoveryengine

    endpoint = "discoveryengine.googleapis.com" if location.lower() == "global" else f"{location}-discoveryengine.googleapis.com"
    client = discoveryengine.SearchServiceClient(client_options=ClientOptions(api_endpoint=endpoint))

    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )

    content_search_spec = discoveryengine.SearchRequest.ContentSearchSpec(
        search_result_mode=discoveryengine.SearchRequest.ContentSearchSpec.SearchResultMode.CHUNKS,
    )

    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=top_k,
        content_search_spec=content_search_spec,
    )

    response = client.search(request=request)

    top_chunks: List[Dict[str, Any]] = []
    for r in response.results:
        chunk = getattr(r, "chunk", None)
        document = getattr(r, "document", None)

        doc_id = (
            str(getattr(document, "id", "") or "").strip()
            or str(getattr(document, "name", "") or "").split("/")[-1].strip()
            or (str(getattr(chunk, "name", "") or "").split("/")[-3].strip() if getattr(chunk, "name", "") else "")
        )

        chunk_id = (
            str(getattr(chunk, "id", "") or "").strip()
            or str(getattr(chunk, "name", "") or "").split("/")[-1].strip()
        )

        content = str(getattr(chunk, "content", "") or "").strip()
        score = float(getattr(r, "relevance_score", 0.0) or 0.0)

        uri = str(getattr(document, "uri", "") or "").strip()
        details: Dict[str, Any] = {}

        if document is not None:
            doc_dict = _proto_to_dict(document)
            if isinstance(doc_dict.get("structData"), dict):
                details.update(doc_dict.get("structData", {}))
            if isinstance(doc_dict.get("derivedStructData"), dict):
                details.update(doc_dict.get("derivedStructData", {}))

        chunk_document_metadata = getattr(chunk, "document_metadata", None) if chunk is not None else None
        if chunk_document_metadata is not None:
            md_dict = _proto_to_dict(chunk_document_metadata)
            if isinstance(md_dict.get("structData"), dict):
                details.update(md_dict.get("structData", {}))
            md_uri = str(md_dict.get("uri", "") or "").strip()
            if md_uri and not uri:
                uri = md_uri

        if not content:
            continue

        if uri:
            details["uri"] = uri

        top_chunks.append(
            {
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "content": content,
                "score": score,
                "uri": uri,
                "document_details": details,
            }
        )

    return top_chunks


def _log_step(project_id: str, step: str, user_id: str, query: str, response: str, citation: Any) -> None:
    payload = {
        "Step": step,
        "User_id": user_id,
        "User_query": query,
        "Response": response,
        "Citation": citation,
    }
    client = gcp_logging.Client(project=project_id)
    client.logger("rag_workflow").log_struct(payload, severity="INFO")


def make_search_tool(project_id: str, location: str, datastore_id: str):
    """Return a search tool function bound to a specific datastore configuration."""
    doc_details_cache: Dict[str, Dict[str, Any]] = {}

    def search_datastore(query: str) -> str:
        """Search the document datastore for relevant chunks matching the query.

        Args:
            query: The search query to find relevant document chunks.

        Returns:
            Formatted document chunks with citation IDs, document IDs, chunk IDs,
            URIs, classification, relevance score, and content.
        """
        chunks = search_relevant_chunks(query, project_id, location, datastore_id)

        if not chunks:
            return "No relevant documents found."

        results = []
        for idx, c in enumerate(chunks, start=1):
            doc_id = c["doc_id"]
            details = c.get("document_details") or get_document_details(
                project_id, location, datastore_id, doc_id, doc_details_cache
            )
            uri = c.get("uri", "") or str(details.get("uri", "") or "")
            classification = _extract_classification_rds(details)

            results.append(
                f"[C{idx}]\n"
                f"doc_id: {doc_id}\n"
                f"chunk_id: {c['chunk_id']}\n"
                f"uri: {uri}\n"
                f"classification_rds: {classification}\n"
                f"score: {c['score']:.4f}\n"
                f"content: {c['content']}"
            )

        return "\n\n".join(results)

    return search_datastore


async def run_rag_agent(
    query: str,
    user_id: str,
    project_id: str,
    location: str,
    datastore_id: str,
    model_location: str = "us-central1",
) -> Dict[str, Any]:
    # Configure ADK to use Vertex AI backend
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
    os.environ["GOOGLE_CLOUD_LOCATION"] = model_location

    search_tool = make_search_tool(project_id, location, datastore_id)

    agent = Agent(
        model="gemini-2.5-flash",
        name="rag_agent",
        instruction=(
            "You are a RAG assistant. Always call the search_datastore tool first to retrieve "
            "relevant information before answering. Answer only from the retrieved contexts. "
            "If the answer is not present in the retrieved contexts, say it is not explicitly available. "
            "Always cite sources inline as [C<number>] using the citation IDs returned by the tool."
        ),
        tools=[search_tool],
    )

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="rag_app", user_id=user_id)
    runner = Runner(agent=agent, app_name="rag_app", session_service=session_service)

    _log_step(project_id, "agent_start", user_id, query, "", [])

    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=genai_types.Content(
            role="user",
            parts=[genai_types.Part(text=query)],
        ),
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        final_response = part.text
                        break

    _log_step(project_id, "agent_complete", user_id, query, final_response, [])

    return {"query": query, "response": final_response}


<!-- ## Sample Run -->


In [15]:
# ── Debug: verify search tool returns results before running the agent ──
import asyncio

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

_debug_search = make_search_tool(project_id, location="us", datastore_id="masked-data_1781603945004")
_debug_result = _debug_search(SAMPLE_QUERY)
print("=== Search Tool Output ===")
print(_debug_result[:3000] if len(_debug_result) > 3000 else _debug_result)


=== Search Tool Output ===
[C1]
doc_id: 606e9fe983494215ce09529219196190
chunk_id: 61
uri: gs://dev-mq-tech-transfer-data-masked/Masked_docs/PPN-AMT-89374_masked.pdf
classification_rds: None
score: 0.0000
content: Number: [DOC_ID_8]

Version: 13.0

Status: Effective Effective Date: 19 Mar 2026
[DOC_ID_8]

Elemental Impurities Risk Assessment – [COMPOUND_1] Injection

Technical Services/Manufacturing Science

The manufacturing equipment and utilities assessments concluded that the equipment does not
contribute Class 1, 2A, or 3 elements because the elements are not present, have a scarce presence
or are controlled. Utilities do not contribute Class 1 or 3 elements, or Cobalt (Co) or Vanadium
(V), because the elements are controlled per regional standards or materials of construction are
neither additive in water, air, nor gases. Nickel (Ni) is present at higher levels in equipment and
utility distribution systems relative to the aforementioned elements; however, this risk assessment
con

In [17]:
# ── Debug: trace all ADK runner events to find where the response is ──
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

_dbg_tool = make_search_tool(project_id, location="us", datastore_id=TARGET_DATASTORE_ID)

_dbg_agent = Agent(
    model="gemini-2.5-flash",
    name="rag_agent_debug",
    instruction=(
        "You are a RAG assistant. Always call the search_datastore tool first to retrieve "
        "relevant information before answering. Answer only from the retrieved contexts. "
        "If the answer is not present in the retrieved contexts, say it is not explicitly available. "
        "Always cite sources inline as [C<number>] using the citation IDs returned by the tool."
    ),
    tools=[_dbg_tool],
)

_dbg_session_service = InMemorySessionService()
_dbg_session = await _dbg_session_service.create_session(app_name="rag_debug", user_id="debug-user")
_dbg_runner = Runner(agent=_dbg_agent, app_name="rag_debug", session_service=_dbg_session_service)

print("=== ADK Event Trace ===")
async for _evt in _dbg_runner.run_async(
    user_id="debug-user",
    session_id=_dbg_session.id,
    new_message=genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=SAMPLE_QUERY)],
    ),
):
    _is_final = _evt.is_final_response()
    _has_content = _evt.content is not None
    _parts_text = []
    if _has_content and _evt.content.parts:
        for _p in _evt.content.parts:
            if _p.text:
                _parts_text.append(repr(_p.text[:120]))
            elif hasattr(_p, "function_call") and _p.function_call:
                _parts_text.append(f"[tool_call: {_p.function_call.name}]")
            elif hasattr(_p, "function_response") and _p.function_response:
                _parts_text.append(f"[tool_response: {_p.function_response.name}]")
    print(f"is_final={_is_final} | has_content={_has_content} | parts={_parts_text}")


=== ADK Event Trace ===
is_final=False | has_content=True | parts=['[tool_call: search_datastore]']
is_final=False | has_content=True | parts=['[tool_response: search_datastore]']
is_final=True | has_content=True | parts=["'The information about product capacity concerns is not explicitly available in the provided contexts. However, one docum'"]


In [16]:
# SAMPLE_QUERY = "In the GLP-1 NPA Phase II stability DIR case study, how many total entries matched exactly after the Power Queries were corrected?"
SAMPLE_QUERY = "Does the product introduce any capacity concerns?"

TARGET_DATASTORE_ID = "masked-data_1781603945004"

result = await run_rag_agent(
    query=SAMPLE_QUERY,
    user_id="sample-user-001",
    project_id=project_id,
    location="us",
    datastore_id=TARGET_DATASTORE_ID,
)

print(result["response"])


Based on the retrieved contexts, there is no explicit information addressing capacity concerns for the product. The documents discuss deliverable volume at manufacturing sites [C1, C5], elemental impurities, and container closure integrity [C2, C3, C5, C7], but they do not mention any limitations or concerns related to overall production capacity or manufacturing output.


In [ ]:
# ── Query Expansion RAG ──
# Replaces answer_query (requires LLM add-on) and plain search (poor recall).
# Strategy: Gemini generates N sub-queries → document-mode search each →
#           deduplicate passages → Gemini answers from the merged context.

from google import genai as _genai
from google.genai import types as _gt
from google.api_core.client_options import ClientOptions as _CO
from google.cloud import discoveryengine_v1 as _de
import re as _re

_llm_client = _genai.Client()


def _expand_queries(original_query: str, n: int = 5) -> List[str]:
    """Use Gemini to generate n search-engine-style sub-queries for better recall."""
    prompt = (
        f"You are helping with document retrieval for a pharmaceutical drug product.\n"
        f'Original question: "{original_query}"\n\n'
        f"Generate {n} short, specific search queries (different phrasings / aspects) "
        f"that together would help retrieve all documents needed to answer the question. "
        f"Output only the queries, one per line, no numbering."
    )
    resp = _llm_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=_gt.GenerateContentConfig(temperature=0.2),
    )
    return [l.strip() for l in (resp.text or "").splitlines() if l.strip()][:n]


def _doc_search_passages(
    query: str, project_id: str, location: str, datastore_id: str, page_size: int = 8
) -> List[Dict]:
    """Document-mode extractive-segment search returning passage dicts."""
    endpoint = f"{location}-discoveryengine.googleapis.com"
    client = _de.SearchServiceClient(client_options=_CO(api_endpoint=endpoint))
    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )
    spec = _de.SearchRequest.ContentSearchSpec(
        search_result_mode=_de.SearchRequest.ContentSearchSpec.SearchResultMode.DOCUMENTS,
        extractive_content_spec=_de.SearchRequest.ContentSearchSpec.ExtractiveContentSpec(
            max_extractive_segment_count=3,
            max_extractive_answer_count=1,
        ),
    )
    req = _de.SearchRequest(
        serving_config=serving_config, query=query,
        page_size=page_size, content_search_spec=spec,
    )
    passages = []
    for r in client.search(req).results:
        doc = r.document
        doc_id = str(getattr(doc, "id", "") or "")
        d = _proto_to_dict(doc)
        derived = d.get("derivedStructData") or {}
        title = str(derived.get("title") or "")
        uri = str(derived.get("link") or derived.get("uri") or "")
        segs = derived.get("extractiveSegments") or derived.get("extractive_segments") or []
        answers = derived.get("extractiveAnswers") or derived.get("extractive_answers") or []
        for seg in (segs or answers):
            content = seg.get("content", "") if isinstance(seg, dict) else str(seg)
            page = seg.get("pageNumber") if isinstance(seg, dict) else None
            if content:
                passages.append({"doc_id": doc_id, "title": title, "uri": uri,
                                  "page": page, "content": content})
    return passages


def query_expansion_rag(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    model: str = "gemini-2.5-flash",
    n_queries: int = 5,
    page_size: int = 8,
) -> Dict[str, Any]:
    """
    RAG with query expansion:
    1. Gemini generates n sub-queries from the original question.
    2. Each sub-query (+ the original) is sent to document-mode search.
    3. Passages are deduplicated by (doc_id, page).
    4. Gemini answers the original question from the merged context.
    """
    # Step 1: expand
    sub_queries = _expand_queries(query, n=n_queries)

    # Step 2: search all queries, deduplicate
    seen: set = set()
    all_passages: List[Dict] = []
    for sq in [query] + sub_queries:
        for p in _doc_search_passages(sq, project_id, location, datastore_id, page_size):
            key = (p["doc_id"], p.get("page"))
            if key not in seen:
                seen.add(key)
                all_passages.append(p)

    if not all_passages:
        return {"query": query, "sub_queries": sub_queries,
                "answer": "No relevant documents found.", "citations": []}

    # Step 3: build context
    context_blocks = []
    for idx, p in enumerate(all_passages, 1):
        page_info = f"\npage: {p['page']}" if p.get("page") else ""
        context_blocks.append(
            f"[C{idx}]\ntitle: {p['title']}\nuri: {p['uri']}{page_info}\ncontent: {p['content']}"
        )

    answer_prompt = (
        "You are a RAG assistant. Answer the question ONLY using the provided contexts.\n"
        "If the answer is not in the contexts, say it is not explicitly available.\n"
        "Cite sources inline as [C<number>].\n\n"
        f"Question: {query}\n\nContexts:\n" + "\n\n".join(context_blocks)
    )

    # Step 4: generate answer
    resp = _llm_client.models.generate_content(
        model=model,
        contents=answer_prompt,
        config=_gt.GenerateContentConfig(temperature=0.1),
    )
    answer_text = resp.text or ""

    # Step 5: extract cited sources
    cited_ids = {int(m) for m in _re.findall(r"\[C(\d+)\]", answer_text)}
    citations = [
        {"index": i, "title": all_passages[i - 1]["title"], "uri": all_passages[i - 1]["uri"]}
        for i in sorted(cited_ids) if 1 <= i <= len(all_passages)
    ]

    return {
        "query": query,
        "sub_queries": sub_queries,
        "passages_retrieved": len(all_passages),
        "answer": answer_text,
        "citations": citations,
    }


# ── Run ──
qe_result = query_expansion_rag(
    query=SAMPLE_QUERY,
    project_id=project_id,
    location="us",
    datastore_id=TARGET_DATASTORE_ID,
)

print(f"Sub-queries used: {qe_result['sub_queries']}")
print(f"Passages retrieved: {qe_result['passages_retrieved']}\n")
print("=== Answer ===")
print(qe_result["answer"])
print(f"\n--- {len(qe_result['citations'])} cited source(s) ---")
for c in qe_result["citations"]:
    print(f"[C{c['index']}] {c['title']}  |  {c['uri']}")


=== Grounded Generation Response ===
Please specify which product you are referring to so I can assess any potential capacity concerns.

--- 0 grounding source(s) ---
